In [ ]:
# ─ patch MJX contact functions before any brax/jit import ─
import smooth_mjx
smooth_mjx.enable(kappa=300.0)

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[smooth_mjx] Enabled sigmoid contact smoothing (kappa=300.0)


In [2]:
import sys
import functools
import optax

import mujoco
from brax import envs
import jax
import jax.numpy as jp

from envs import register_ahac_anymal
import shac.networks as shac_networks
from shac.train import SHAC

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
env_name = 'ahac_anymal'

# Schwarke et al.: (128, 64, 32) actor MLP, (64, 64) critic MLP, ELU activation
make_networks_factory = functools.partial(
    shac_networks.make_shac_networks,
    policy_hidden_layer_sizes=(128, 64, 32),
    value_hidden_layer_sizes=(64, 64),
    scalar_var=False,
    layer_norm=True,
)

In [4]:
# ── Training hyperparameters (Schwarke et al. §A.3) ─────────────────────────
unroll_length = 32   # 32 × 0.02s = 0.64s ≈ 1.3 trot cycles (2Hz gait, 25 steps/cycle)
num_envs      = 512
episode_length = 1000  # 1000 × 0.02s = 20s

# 40,000 gradient updates (2× v9 — policy was still trending up)
num_training_steps = 40_000
num_timesteps  = num_training_steps * num_envs * unroll_length
num_evals      = 200

num_critic_minibatches = 4
critic_batch_size = (num_envs * unroll_length) // num_critic_minibatches

# LR schedules decay over the full training run
_actor_lr  = optax.cosine_decay_schedule(init_value=5e-3, decay_steps=num_training_steps, alpha=1e-5/5e-3)
_critic_lr = optax.cosine_decay_schedule(init_value=2e-3, decay_steps=num_training_steps, alpha=1e-5/2e-3)

env_kwargs = {
    "termination_height": 0.25,
    "physics_steps_per_control_step": 4,    # 4 × 0.005s = 0.02s control dt (Schwarke)
    "model_variant": "anymal",
    "smooth_sigma_q": 0.0,
    "smooth_sigma_v": 0.0,
    "use_domain_randomization": True,
}
eval_env_kwargs = {**env_kwargs, "use_domain_randomization": False}

In [5]:
env      = envs.get_environment(env_name, **env_kwargs)
eval_env = envs.get_environment(env_name, **eval_env_kwargs)

print(f"Obs size: {env.observation_size}  |  Action size: {env.action_size}")


Obs size: (49,)  |  Action size: 12


In [ ]:
trainer = SHAC(
    environment=env,
    eval_env=eval_env,
    num_timesteps=num_timesteps,
    episode_length=episode_length,
    num_envs=num_envs,
    num_eval_envs=64,
    unroll_length=unroll_length,
    critic_batch_size=critic_batch_size,
    critic_epochs=16,
    target_critic_alpha=0.995,
    discounting=0.99,
    lambda_=0.95,
    normalize_observations=True,
    reward_scaling=1.0,
    network_factory=make_networks_factory,
    actor_learning_rate=_actor_lr,
    critic_learning_rate=_critic_lr,
    entropy_cost=0.0,
    seed=0,
    num_evals=num_evals,
    use_tbx=True,
    tbx_logdir=f'{env_name}_log',
    tbx_experiment_name="v10",
    resample_init=True,
    scramble_initial_times=True,
    save_all_checkpoints=False,
    checkpoint_every=50,
    polgrad_thresh=1e6,
    grad_clip_norm=1.0,
)

Env steps per training step: 16384
Training steps per epoch: 202
Critic minibatches per critic epoch: 4


In [7]:
make_inference_fn, policy_params, value_params, _ = trainer.train()


Initial eval time: 53.1910 s
Deleting old checkpoints!
Checkpointed for epoch 0
Checkpointed for epoch 50
Checkpointed for epoch 100
Checkpointed for epoch 150
Checkpointed for epoch 198


In [ ]:
import pickle, pathlib
pathlib.Path("saved_policies").mkdir(exist_ok=True)
_save_path = "saved_policies/v10.pkl"
with open(_save_path, "wb") as f:
    pickle.dump(policy_params, f)
print(f"stage 1 params saved → {_save_path}")

Stage 1 params saved → saved_policies/schwarke_stage1_v10.pkl
